# Train ARIMA and SARIMA time-series models, host inference on a Docker container running on Amazon ECS on AWS Fargate

This notebook demonstrates how to train ARIMA and SARIMA (Seasonal ARIMA) time-series forecasting models using the [California Housing dataset](https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html), package the trained model into a Docker container, deploy it on [Amazon ECS on AWS Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html), and optionally expose it as an API with [Amazon API Gateway](https://aws.amazon.com/api-gateway/).

**What is ARIMA?**

ARIMA (AutoRegressive Integrated Moving Average) is a statistical model for analyzing and forecasting time-series data. It combines:
- **AR (AutoRegressive):** Past values influence future values
- **I (Integrated):** Differencing to make the series stationary
- **MA (Moving Average):** Past forecast errors influence future values

ARIMA is specified by three parameters: `(p, d, q)` where p = AR order, d = differencing order, q = MA order.

**What is SARIMA?**

SARIMA (Seasonal ARIMA) extends ARIMA by adding seasonal components: `(p, d, q) x (P, D, Q, s)` where the uppercase letters represent the seasonal counterparts and `s` is the seasonal period (e.g., 12 for monthly data with annual seasonality).

**Adapting the California Housing dataset for time-series:**

The California Housing dataset is cross-sectional (not inherently a time-series). To demonstrate ARIMA/SARIMA, we aggregate the `median_house_value` by `housing_median_age` bins to create a synthetic time-series representing how median house values change across housing age cohorts. This simulates a temporal trend suitable for time-series modeling.

**Note:**
- ARIMA/SARIMA training is CPU-based — there is no GPU acceleration benefit for these models.
- Training is done locally in this notebook (no SageMaker training job required) since statsmodels is lightweight.
- The deployed container uses the same ECS Fargate infrastructure pattern as the XGBoost notebook.

**Table of Contents:**

1. [Complete prerequisites](#Complete-prerequisites)
    1. [Check and configure access to the Internet](#Check-and-configure-access-to-the-Internet)
    2. [Check and upgrade required software versions](#Check-and-upgrade-required-software-versions)
    3. [Check and configure security permissions](#Check-and-configure-security-permissions)
    4. [Organize imports](#Organize-imports)
    5. [Create common objects](#Create-common-objects)
2. [Prepare the data](#Prepare-the-data)
    1. [Load the dataset](#Load-the-dataset)
    2. [Create the time-series](#Create-the-time-series)
    3. [Visualize the time-series](#Visualize-the-time-series)
    4. [Test for stationarity](#Test-for-stationarity)
    5. [Split into train and test sets](#Split-into-train-and-test-sets)
3. [Train the ARIMA model](#Train-the-ARIMA-model)
    1. [Determine ARIMA parameters](#Determine-ARIMA-parameters)
    2. [Fit the ARIMA model](#Fit-the-ARIMA-model)
    3. [Evaluate the ARIMA model](#Evaluate-the-ARIMA-model)
4. [Train the SARIMA model](#Train-the-SARIMA-model)
    1. [Determine SARIMA parameters](#Determine-SARIMA-parameters)
    2. [Fit the SARIMA model](#Fit-the-SARIMA-model)
    3. [Evaluate the SARIMA model](#Evaluate-the-SARIMA-model)
    4. [Compare ARIMA vs SARIMA](#Compare-ARIMA-vs-SARIMA)
5. [Save and export the model](#Save-and-export-the-model)
6. [Create and push the Docker container to Amazon ECR](#Create-and-push-the-Docker-container-to-Amazon-ECR)
    1. [View the inference script](#View-the-inference-script)
    2. [Create the Dockerfile](#Create-the-Dockerfile)
    3. [Create the container](#Create-the-container)
    4. [Create the private repository in ECR](#Create-the-private-repository-in-ECR)
    5. [Push the container to ECR](#Push-the-container-to-ECR)
7. [Deploy and test on Amazon ECS on AWS Fargate](#Deploy-and-test-on-Amazon-ECS-on-AWS-Fargate)
    1. [Create the ECS cluster](#Create-the-ECS-cluster)
    2. [Create the ECS Task and deploy the container](#Create-the-ECS-Task-and-deploy-the-container)
    3. [Prepare to test the ECS Task](#Prepare-to-test-the-ECS-Task)
    4. [Test the ECS Task](#Test-the-ECS-Task)
8. [Cleanup](#Cleanup)
    1. [Cleanup ECS resources](#Cleanup-ECS-resources)
    2. [Cleanup ECR repository](#Cleanup-ECR-repository)

## 1. Complete prerequisites <a id='Complete-prerequisites'></a>

### A) Check and configure access to the Internet <a id='Check-and-configure-access-to-the-Internet'></a>

This notebook requires outbound access to the Internet to download software and to make calls to the ECS Task. You can provide direct Internet access (default) or through a VPC.

### B) Check and upgrade required software versions <a id='Check-and-upgrade-required-software-versions'></a>

This notebook requires:
- Python 3.8+
- statsmodels (for ARIMA/SARIMA)
- pandas, numpy, matplotlib, seaborn
- boto3
- Docker with BuildKit/buildx plugin
- AWS CLI
- cURL

In [ ]:
import platform
import os

def get_os_version():
    """Detect the notebook environment OS."""
    if os.path.exists('/etc/os-release'):
        with open('/etc/os-release') as f:
            content = f.read()
        if 'Ubuntu' in content:
            return 'Ubuntu'
        elif 'Amazon Linux 2023' in content or 'AL2023' in content:
            return 'AL2023'
        elif 'Amazon Linux 2' in content:
            return 'ALv2'
        elif 'Amazon Linux' in content:
            return 'ALv1'
    return f"Unknown ({platform.platform()})"

os_version = get_os_version()
print(f"Notebook OS version: {os_version}")

In [ ]:
import sys
import subprocess

# Install required packages
packages = ['statsmodels', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'flask', 'boto3']
for pkg in packages:
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', pkg])

import boto3
import statsmodels
import pandas as pd
import numpy as np

print(f'Python version: {sys.version}')
print(f'statsmodels version: {statsmodels.__version__}')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')
print(f'boto3 version: {boto3.__version__}')

# AWS CLI
!aws --version

In [ ]:
# Verify Docker is installed
!docker --version

# Install docker-buildx plugin if needed
!mkdir -p ~/.docker/cli-plugins
!ARCH=$(uname -m) && \
  if [ "$ARCH" = "x86_64" ]; then ARCH="amd64"; elif [ "$ARCH" = "aarch64" ]; then ARCH="arm64"; fi && \
  curl -sSL "https://github.com/docker/buildx/releases/download/v0.36.1/buildx-v0.36.1.linux-${ARCH}" -o ~/.docker/cli-plugins/docker-buildx && \
  chmod +x ~/.docker/cli-plugins/docker-buildx

!docker buildx version

In [ ]:
# Install and configure the Amazon ECR credential helper
if 'Ubuntu' in os_version:
    !sudo apt-get update -qq && sudo apt-get install -y -qq amazon-ecr-credential-helper
elif os_version in ('ALv2', 'AL2023'):
    !sudo yum --assumeyes install amazon-ecr-credential-helper
elif os_version == 'ALv1':
    print('ECR credential helper not needed on ALv1.')

if os_version != 'ALv1':
    !docker-credential-ecr-login version
    !mkdir -p ~/.docker
    !printf '{\n\t"credsStore": "ecr-login"\n}' > ~/.docker/config.json
    !cat ~/.docker/config.json

### C) Check and configure security permissions <a id='Check-and-configure-security-permissions'></a>

This notebook uses the IAM execution role attached to the underlying SageMaker environment. This role should have:
1. Access to create CloudWatch Log Groups and write logs.
2. Access to create, delete and write to Amazon ECR private registries.
3. Access to create and delete Amazon ECS clusters and task definitions.
4. Access to run ECS tasks.
5. Access to describe EC2 network interfaces (to retrieve task public IP).

In [ ]:
# Print the execution role (if running on SageMaker)
try:
    from sagemaker.core.helper.session_helper import get_execution_role
    print(f'Execution role: {get_execution_role()}')
except Exception:
    print('Not running on SageMaker or sagemaker SDK not available.')
    print('Ensure your AWS credentials have the required permissions.')

### D) Organize imports <a id='Organize-imports'></a>

In [ ]:
import json
import logging
import os
import pickle
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import boto3

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('All imports successful.')

### E) Create common objects <a id='Create-common-objects'></a>

In [ ]:
# AWS clients
region_name = boto3.session.Session().region_name
if not region_name:
    region_name = 'us-east-1'  # fallback
print(f'AWS Region: {region_name}')

ecs_client = boto3.client('ecs', region_name=region_name)
ecr_client = boto3.client('ecr', region_name=region_name)
ec2_client = boto3.client('ec2', region_name=region_name)
sts_client = boto3.client('sts', region_name=region_name)

# Account ID
account_id = sts_client.get_caller_identity()['Account']
print(f'AWS Account ID: {account_id}')

# Notebook name (used as prefix for resources)
nb_name = 'sarima-arima-ca-housing-ecs-container-model-hosting'

# Container configuration
container_image_name = nb_name
container_image_tag = 'latest'
container_artifacts_dir = 'container-artifacts'

# ECS configuration
ecs_cluster_name = f'{nb_name}-cluster'
ecs_fargate_task_name = f'fargate-task-{nb_name}'
ecs_container_name = f'{nb_name}-container'
ecs_container_port = 80
ecs_container_host_port = 80
ecs_fargate_task_cpu = '512'
ecs_fargate_task_memory = '1024'
ecs_fargate_task_count = 1

# Health check
ecs_container_healthcheck_command_list = ['CMD-SHELL', f'curl -f http://localhost:{ecs_container_port}/healthcheck || exit 1']
ecs_container_healthcheck_interval_in_seconds = 30
ecs_container_healthcheck_timeout_in_seconds = 5

print(f'\nNotebook name: {nb_name}')
print(f'ECS Cluster: {ecs_cluster_name}')
print(f'Container image: {container_image_name}:{container_image_tag}')

In [ ]:
# =============================================================================
# ECS TASK IAM ROLES - CONFIGURE THESE
# =============================================================================
# The ECS Task Execution Role needs:
#   - AmazonECSTaskExecutionRolePolicy (managed policy)
#   - CloudWatch Logs CreateLogGroup permission
#
# The ECS Task Role needs:
#   - Any permissions the container itself requires at runtime
#
# Replace with your role ARNs:
ecs_fargate_task_execution_role = f'arn:aws:iam::{account_id}:role/ecsTaskExecutionRole'
ecs_fargate_task_role = f'arn:aws:iam::{account_id}:role/ecsTaskRole'

print(f'Task Execution Role: {ecs_fargate_task_execution_role}')
print(f'Task Role: {ecs_fargate_task_role}')
print()
print('NOTE: Update the role ARNs above if they differ in your account.')

In [ ]:
# =============================================================================
# ECS NETWORKING - CONFIGURE THESE
# =============================================================================
# Specify the VPC subnet(s) and security group(s) for the ECS Fargate task.
# The subnet must be public (or have a NAT Gateway) for the task to pull
# the container image from ECR and receive inbound traffic.
#
# Replace with your values:
ecs_fargate_task_subnet_list = ['<subnet-id>']  # e.g., ['subnet-0abc123def456']
ecs_fargate_task_security_group_list = ['<security-group-id>']  # e.g., ['sg-0abc123def456']

print(f'Subnets: {ecs_fargate_task_subnet_list}')
print(f'Security Groups: {ecs_fargate_task_security_group_list}')
print()
print('NOTE: Update the subnet and security group IDs for your VPC.')
print('The security group must allow inbound TCP on port 80.')

## 2. Prepare the data <a id='Prepare-the-data'></a>

We transform the California Housing cross-sectional dataset into a time-series by aggregating median house values by `housing_median_age`. This creates a series indexed by housing age (1–52 years), representing the average median house value for each age cohort — a synthetic temporal dimension.

### A) Load the dataset <a id='Load-the-dataset'></a>

In [ ]:
# Load the California Housing dataset
dataset_path = 'datasets/california_housing.csv'
df = pd.read_csv(dataset_path)

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

### B) Create the time-series <a id='Create-the-time-series'></a>

We aggregate `median_house_value` by `housing_median_age` to create a time-series. Each point represents the mean of all median house values for a given housing age cohort.

In [ ]:
# Aggregate: mean median_house_value grouped by housing_median_age
ts_df = df.groupby('housing_median_age')['median_house_value'].mean().reset_index()
ts_df.columns = ['age', 'value']
ts_df = ts_df.sort_values('age').reset_index(drop=True)

# Create a proper time-series index (treat age as sequential periods)
# We use a monthly frequency starting from a base date for statsmodels compatibility
ts_df['date'] = pd.date_range(start='2000-01-01', periods=len(ts_df), freq='MS')
ts_series = ts_df.set_index('date')['value']

print(f'Time-series length: {len(ts_series)} observations')
print(f'Date range: {ts_series.index[0]} to {ts_series.index[-1]}')
print(f'\nDescriptive statistics:')
print(ts_series.describe())
print(f'\nFirst 10 values:')
ts_series.head(10)

### C) Visualize the time-series <a id='Visualize-the-time-series'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time-series plot
axes[0, 0].plot(ts_series.index, ts_series.values, 'b-', linewidth=1.5)
axes[0, 0].set_title('Median House Value by Housing Age Cohort')
axes[0, 0].set_xlabel('Date (proxy for housing age)')
axes[0, 0].set_ylabel('Mean Median House Value ($)')
axes[0, 0].grid(True, alpha=0.3)

# Distribution
axes[0, 1].hist(ts_series.values, bins=15, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution of Values')
axes[0, 1].set_xlabel('Value ($)')
axes[0, 1].set_ylabel('Frequency')

# ACF
plot_acf(ts_series, lags=min(20, len(ts_series)//2 - 1), ax=axes[1, 0])
axes[1, 0].set_title('Autocorrelation Function (ACF)')

# PACF
plot_pacf(ts_series, lags=min(15, len(ts_series)//2 - 1), ax=axes[1, 1])
axes[1, 1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

### D) Test for stationarity <a id='Test-for-stationarity'></a>

ARIMA requires the series to be stationary (or made stationary through differencing). We use the Augmented Dickey-Fuller (ADF) test.

In [ ]:
def adf_test(series, name=''):
    """Perform the Augmented Dickey-Fuller test for stationarity."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Test Results {name}')
    print(f'  Test Statistic:  {result[0]:.4f}')
    print(f'  p-value:         {result[1]:.4f}')
    print(f'  Lags Used:       {result[2]}')
    print(f'  Observations:    {result[3]}')
    print(f'  Critical Values:')
    for key, value in result[4].items():
        print(f'    {key}: {value:.4f}')
    if result[1] <= 0.05:
        print(f'  Conclusion: Series IS stationary (p <= 0.05)')
    else:
        print(f'  Conclusion: Series is NOT stationary (p > 0.05) — differencing needed')
    print()
    return result[1]

# Test original series
p_value_orig = adf_test(ts_series, '(Original Series)')

# Test first difference
ts_diff = ts_series.diff().dropna()
p_value_diff = adf_test(ts_diff, '(First Difference)')

# Determine d parameter
d = 0 if p_value_orig <= 0.05 else 1
print(f'Recommended differencing order (d): {d}')

### E) Split into train and test sets <a id='Split-into-train-and-test-sets'></a>

In [ ]:
# Use 80% for training, 20% for testing
split_idx = int(len(ts_series) * 0.8)
train_series = ts_series[:split_idx]
test_series = ts_series[split_idx:]

print(f'Training set: {len(train_series)} observations ({train_series.index[0]} to {train_series.index[-1]})')
print(f'Test set:     {len(test_series)} observations ({test_series.index[0]} to {test_series.index[-1]})')

# Visualize the split
plt.figure(figsize=(12, 5))
plt.plot(train_series.index, train_series.values, 'b-', label='Train', linewidth=1.5)
plt.plot(test_series.index, test_series.values, 'r-', label='Test', linewidth=1.5)
plt.axvline(x=train_series.index[-1], color='gray', linestyle='--', alpha=0.7)
plt.title('Train/Test Split')
plt.xlabel('Date')
plt.ylabel('Mean Median House Value ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Train the ARIMA model <a id='Train-the-ARIMA-model'></a>

ARIMA(p, d, q) where:
- p = number of autoregressive terms
- d = number of differences needed for stationarity
- q = number of moving average terms

### A) Determine ARIMA parameters <a id='Determine-ARIMA-parameters'></a>

We use a grid search approach to find the best (p, d, q) based on AIC (Akaike Information Criterion).

In [ ]:
# Grid search for best ARIMA parameters
best_aic = float('inf')
best_arima_order = None
arima_results = []

p_range = range(0, 5)
d_range = [d]  # Use the d determined from ADF test
q_range = range(0, 5)

print('Searching for best ARIMA(p,d,q) parameters...')
print(f'Search space: p={list(p_range)}, d={list(d_range)}, q={list(q_range)}')
print()

for p in p_range:
    for d_val in d_range:
        for q in q_range:
            try:
                model = ARIMA(train_series, order=(p, d_val, q))
                fitted = model.fit()
                aic = fitted.aic
                arima_results.append({'p': p, 'd': d_val, 'q': q, 'AIC': aic})
                if aic < best_aic:
                    best_aic = aic
                    best_arima_order = (p, d_val, q)
            except Exception:
                continue

# Display top 5 results
arima_results_df = pd.DataFrame(arima_results).sort_values('AIC').head(10)
print(f'Best ARIMA order: {best_arima_order} (AIC = {best_aic:.2f})')
print(f'\nTop 10 ARIMA configurations by AIC:')
arima_results_df

### B) Fit the ARIMA model <a id='Fit-the-ARIMA-model'></a>

In [ ]:
# Fit the best ARIMA model
print(f'Fitting ARIMA{best_arima_order} model...')
arima_model = ARIMA(train_series, order=best_arima_order)
arima_fitted = arima_model.fit()

print(f'\nModel Summary:')
print(arima_fitted.summary())

### C) Evaluate the ARIMA model <a id='Evaluate-the-ARIMA-model'></a>

In [ ]:
# Generate forecasts for the test period
n_test = len(test_series)
arima_forecast = arima_fitted.get_forecast(steps=n_test)
arima_pred = arima_forecast.predicted_mean
arima_conf_int = arima_forecast.conf_int()

# Calculate metrics
arima_mae = mean_absolute_error(test_series.values, arima_pred.values)
arima_rmse = np.sqrt(mean_squared_error(test_series.values, arima_pred.values))
arima_mape = np.mean(np.abs((test_series.values - arima_pred.values) / test_series.values)) * 100

print(f'ARIMA{best_arima_order} Evaluation on Test Set:')
print(f'  MAE:  ${arima_mae:,.2f}')
print(f'  RMSE: ${arima_rmse:,.2f}')
print(f'  MAPE: {arima_mape:.2f}%')

# Plot
plt.figure(figsize=(12, 6))
plt.plot(train_series.index, train_series.values, 'b-', label='Train', linewidth=1.5)
plt.plot(test_series.index, test_series.values, 'g-', label='Actual (Test)', linewidth=1.5)
plt.plot(test_series.index, arima_pred.values, 'r--', label=f'ARIMA{best_arima_order} Forecast', linewidth=1.5)
plt.fill_between(test_series.index,
                 arima_conf_int.iloc[:, 0],
                 arima_conf_int.iloc[:, 1],
                 alpha=0.2, color='red', label='95% Confidence Interval')
plt.title(f'ARIMA{best_arima_order} Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Mean Median House Value ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Train the SARIMA model <a id='Train-the-SARIMA-model'></a>

SARIMA(p, d, q)(P, D, Q, s) adds seasonal components. We use `s=12` assuming monthly seasonality (annual cycle) in our synthetic time-series.

### A) Determine SARIMA parameters <a id='Determine-SARIMA-parameters'></a>

We search over seasonal parameters while using the best ARIMA order as the non-seasonal component.

In [ ]:
# Seasonal period
seasonal_period = 12  # Annual seasonality with monthly data

# Adjust seasonal period if our series is shorter
if len(train_series) < 2 * seasonal_period:
    seasonal_period = max(4, len(train_series) // 4)
    print(f'Adjusted seasonal period to {seasonal_period} (series too short for s=12)')

# Grid search for seasonal parameters
best_sarima_aic = float('inf')
best_sarima_order = None
best_seasonal_order = None
sarima_results = []

P_range = range(0, 3)
D_range = range(0, 2)
Q_range = range(0, 3)

print(f'Searching for best SARIMA parameters...')
print(f'Non-seasonal order: {best_arima_order}')
print(f'Seasonal search space: P={list(P_range)}, D={list(D_range)}, Q={list(Q_range)}, s={seasonal_period}')
print()

for P in P_range:
    for D in D_range:
        for Q in Q_range:
            try:
                model = SARIMAX(train_series,
                               order=best_arima_order,
                               seasonal_order=(P, D, Q, seasonal_period),
                               enforce_stationarity=False,
                               enforce_invertibility=False)
                fitted = model.fit(disp=False, maxiter=200)
                aic = fitted.aic
                sarima_results.append({
                    'order': best_arima_order,
                    'P': P, 'D': D, 'Q': Q, 's': seasonal_period,
                    'AIC': aic
                })
                if aic < best_sarima_aic:
                    best_sarima_aic = aic
                    best_sarima_order = best_arima_order
                    best_seasonal_order = (P, D, Q, seasonal_period)
            except Exception:
                continue

sarima_results_df = pd.DataFrame(sarima_results).sort_values('AIC').head(10)
print(f'Best SARIMA: order={best_sarima_order}, seasonal_order={best_seasonal_order} (AIC = {best_sarima_aic:.2f})')
print(f'\nTop 10 SARIMA configurations by AIC:')
sarima_results_df

### B) Fit the SARIMA model <a id='Fit-the-SARIMA-model'></a>

In [ ]:
# Fit the best SARIMA model on the FULL training series
print(f'Fitting SARIMA{best_sarima_order}x{best_seasonal_order} model...')
sarima_model = SARIMAX(train_series,
                       order=best_sarima_order,
                       seasonal_order=best_seasonal_order,
                       enforce_stationarity=False,
                       enforce_invertibility=False)
sarima_fitted = sarima_model.fit(disp=False, maxiter=200)

print(f'\nModel Summary:')
print(sarima_fitted.summary())

### C) Evaluate the SARIMA model <a id='Evaluate-the-SARIMA-model'></a>

In [ ]:
# Generate forecasts for the test period
sarima_forecast = sarima_fitted.get_forecast(steps=n_test)
sarima_pred = sarima_forecast.predicted_mean
sarima_conf_int = sarima_forecast.conf_int()

# Calculate metrics
sarima_mae = mean_absolute_error(test_series.values, sarima_pred.values)
sarima_rmse = np.sqrt(mean_squared_error(test_series.values, sarima_pred.values))
sarima_mape = np.mean(np.abs((test_series.values - sarima_pred.values) / test_series.values)) * 100

print(f'SARIMA{best_sarima_order}x{best_seasonal_order} Evaluation on Test Set:')
print(f'  MAE:  ${sarima_mae:,.2f}')
print(f'  RMSE: ${sarima_rmse:,.2f}')
print(f'  MAPE: {sarima_mape:.2f}%')

# Plot
plt.figure(figsize=(12, 6))
plt.plot(train_series.index, train_series.values, 'b-', label='Train', linewidth=1.5)
plt.plot(test_series.index, test_series.values, 'g-', label='Actual (Test)', linewidth=1.5)
plt.plot(test_series.index, sarima_pred.values, 'r--', label=f'SARIMA Forecast', linewidth=1.5)
plt.fill_between(test_series.index,
                 sarima_conf_int.iloc[:, 0],
                 sarima_conf_int.iloc[:, 1],
                 alpha=0.2, color='red', label='95% Confidence Interval')
plt.title(f'SARIMA{best_sarima_order}x{best_seasonal_order} Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Mean Median House Value ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### D) Compare ARIMA vs SARIMA <a id='Compare-ARIMA-vs-SARIMA'></a>

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    'Metric': ['MAE ($)', 'RMSE ($)', 'MAPE (%)', 'AIC'],
    f'ARIMA{best_arima_order}': [
        f'{arima_mae:,.2f}', f'{arima_rmse:,.2f}',
        f'{arima_mape:.2f}', f'{best_aic:.2f}'
    ],
    f'SARIMA{best_sarima_order}x{best_seasonal_order}': [
        f'{sarima_mae:,.2f}', f'{sarima_rmse:,.2f}',
        f'{sarima_mape:.2f}', f'{best_sarima_aic:.2f}'
    ]
})
print('Model Comparison:')
print(comparison.to_string(index=False))

# Select the best model for deployment
if sarima_rmse <= arima_rmse:
    best_model = sarima_fitted
    best_model_name = f'SARIMA{best_sarima_order}x{best_seasonal_order}'
else:
    best_model = arima_fitted
    best_model_name = f'ARIMA{best_arima_order}'

print(f'\nBest model for deployment: {best_model_name}')

# Plot both forecasts
plt.figure(figsize=(14, 6))
plt.plot(train_series.index, train_series.values, 'b-', label='Train', linewidth=1.5)
plt.plot(test_series.index, test_series.values, 'g-', label='Actual (Test)', linewidth=2)
plt.plot(test_series.index, arima_pred.values, 'r--', label=f'ARIMA{best_arima_order}', linewidth=1.5)
plt.plot(test_series.index, sarima_pred.values, 'm--', label=f'SARIMA{best_sarima_order}x{best_seasonal_order}', linewidth=1.5)
plt.title('ARIMA vs SARIMA Forecast Comparison')
plt.xlabel('Date')
plt.ylabel('Mean Median House Value ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Save and export the model <a id='Save-and-export-the-model'></a>

We retrain the best model on the **full** dataset (train + test) for production deployment, then save it as a pickle file.

In [ ]:
# Retrain on full series for production
print(f'Retraining {best_model_name} on full dataset ({len(ts_series)} observations)...')

if 'SARIMA' in best_model_name:
    production_model = SARIMAX(ts_series,
                               order=best_sarima_order,
                               seasonal_order=best_seasonal_order,
                               enforce_stationarity=False,
                               enforce_invertibility=False)
else:
    production_model = ARIMA(ts_series, order=best_arima_order)

production_fitted = production_model.fit(disp=False, maxiter=200)
print('Training complete.')
print(f'AIC: {production_fitted.aic:.2f}')

In [ ]:
# Create the container artifacts directory
os.makedirs(container_artifacts_dir, exist_ok=True)

# Save the model as a pickle file
model_pickle_path = os.path.join(container_artifacts_dir, 'model.pkl')
with open(model_pickle_path, 'wb') as f:
    pickle.dump(production_fitted, f)

model_size_mb = os.path.getsize(model_pickle_path) / (1024 * 1024)
print(f'Model saved to: {model_pickle_path}')
print(f'Model size: {model_size_mb:.2f} MB')

# Verify the saved model
with open(model_pickle_path, 'rb') as f:
    loaded_model = pickle.load(f)

verify_forecast = loaded_model.get_forecast(steps=3)
print(f'\nVerification — Next 3 forecasts: {verify_forecast.predicted_mean.values}')
print('Model pickle verified successfully.')

### (Stage) Copy the trained model to the inference app (ECS worker) <a id='Stage-the-trained-model'></a>
Copy the saved production pickle into `app/backend/ecs_worker/models/` using the file name the ECS worker expects for the route it serves (`sarima-model.pkl` or `arima-model.pkl`). This automates the manual staging step in `instructions.md` so the model is ready for the worker image build. The staged file is git-ignored.


In [ ]:
# Stage the saved time-series model for the containerized inference app.
# The ECS worker loads '<model-name>-model.pkl' from /opt/models (baked into the image
# from app/backend/ecs_worker/models/). This automates the manual staging step.
import shutil

# Resolve the app's model staging directory relative to this notebook.
# Override ECS_WORKER_MODELS_DIR if your repository layout differs.
ecs_worker_models_dir = os.environ.get(
    'ECS_WORKER_MODELS_DIR',
    os.path.join(os.getcwd(), '..', 'app', 'backend', 'ecs_worker', 'models'),
)
ecs_worker_models_dir = os.path.abspath(ecs_worker_models_dir)
os.makedirs(ecs_worker_models_dir, exist_ok=True)

# Name the staged pickle after the model route it serves (arima or sarima),
# based on the best model retrained on the full series above.
staged_model_route = 'sarima' if 'SARIMA' in best_model_name else 'arima'
staged_model_path = os.path.join(ecs_worker_models_dir, f'{staged_model_route}-model.pkl')
shutil.copyfile(model_pickle_path, staged_model_path)

staged_size_mb = os.path.getsize(staged_model_path) / (1024 * 1024)
print(f'Staged {best_model_name} model for the ECS worker image:')
print(f'  source: {model_pickle_path}')
print(f'  target: {staged_model_path} ({staged_size_mb:.2f} MB)')
print(f"\nThe '{staged_model_route}' route will serve this trained model. The other "
      "route uses the built-in naive-forecast fallback unless you stage its pickle too.")
print('This file is git-ignored. Next: build and push the worker image '
      '(instructions.md step 3.2).')


## 6. Create and push the Docker container to Amazon ECR <a id='Create-and-push-the-Docker-container-to-Amazon-ECR'></a>

We package the trained model into a Docker container with a Flask inference server and push it to Amazon ECR.

### A) View the inference script <a id='View-the-inference-script'></a>

The SARIMA inference script is a Flask app that:
- Loads the model pickle at startup
- Accepts POST requests with `{"steps": N}` to generate N-step forecasts
- Returns forecast values with confidence intervals
- Implements a `/healthcheck` endpoint for ECS

In [ ]:
# View the SARIMA inference script
inference_script_path = 'scripts/container_sarima_inference.py'
!cat {inference_script_path}

### B) Create the Dockerfile <a id='Create-the-Dockerfile'></a>

In [ ]:
# Copy inference script and requirements to container-artifacts
!cp -pr scripts/container_sarima_inference.py {container_artifacts_dir}/server.py
!cp -pr scripts/container_sarima_inference_requirements.txt {container_artifacts_dir}/requirements.txt

# Create the Dockerfile
dockerfile_content = f"""# SARIMA/ARIMA inference container
FROM public.ecr.aws/amazonlinux/amazonlinux:2023

WORKDIR /app

# Install Python 3, pip, and curl (for health checks)
RUN dnf -y install python3 python3-pip curl && dnf clean all

# Create virtual environment
RUN python3 -m venv /opt/appenv

# Install Python dependencies
COPY requirements.txt .
RUN /opt/appenv/bin/pip install --no-cache-dir --upgrade pip && \\
    /opt/appenv/bin/pip install --no-cache-dir -r requirements.txt

# Copy model and inference server
COPY model.pkl ./model.pkl
COPY server.py ./server.py

# Environment variables
ENV MODEL_PICKLE_FILE_PATH=/app/model.pkl
ENV FLASK_SERVER_LOG_LEVEL=INFO
ENV FLASK_SERVER_HOSTNAME=0.0.0.0
ENV FLASK_SERVER_PORT=80
ENV FLASK_SERVER_DEBUG=False

EXPOSE 80

HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD curl -f http://localhost:80/healthcheck || exit 1

ENTRYPOINT ["/opt/appenv/bin/python", "server.py"]
"""

dockerfile_path = os.path.join(container_artifacts_dir, 'Dockerfile')
with open(dockerfile_path, 'w') as f:
    f.write(dockerfile_content)

print(f'Dockerfile created at: {dockerfile_path}')
print()
print(dockerfile_content)

### C) Create the container <a id='Create-the-container'></a>

In [ ]:
# Build the Docker container
!docker build -t {container_image_name}:{container_image_tag} {container_artifacts_dir}/

print(f'\nDocker image built: {container_image_name}:{container_image_tag}')
!docker images {container_image_name}

### D) Create the private repository in ECR <a id='Create-the-private-repository-in-ECR'></a>

In [ ]:
# Create ECR repository (ignore error if it already exists)
try:
    ecr_client.create_repository(
        repositoryName=container_image_name,
        imageScanningConfiguration={'scanOnPush': True}
    )
    print(f'ECR repository created: {container_image_name}')
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f'ECR repository already exists: {container_image_name}')

# Get the repository URI
ecr_repo_uri = f'{account_id}.dkr.ecr.{region_name}.amazonaws.com/{container_image_name}'
target_image_name = f'{ecr_repo_uri}:{container_image_tag}'
print(f'Target image: {target_image_name}')

### E) Push the container to ECR <a id='Push-the-container-to-ECR'></a>

In [ ]:
# Tag and push the image
!docker tag {container_image_name}:{container_image_tag} {target_image_name}

# Login to ECR (for ALv1 environments without credential helper)
if os_version == 'ALv1':
    !aws ecr get-login-password --region {region_name} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region_name}.amazonaws.com

# Push
!docker push {target_image_name}

print(f'\nImage pushed to ECR: {target_image_name}')

## 7. Deploy and test on Amazon ECS on AWS Fargate <a id='Deploy-and-test-on-Amazon-ECS-on-AWS-Fargate'></a>

### A) Create the ECS cluster <a id='Create-the-ECS-cluster'></a>

In [ ]:
# Create the ECS cluster
try:
    describe_ecs_cluster_response = ecs_client.describe_clusters(clusters=[ecs_cluster_name])
    if not describe_ecs_cluster_response['clusters'] or \
       describe_ecs_cluster_response['clusters'][0]['status'] == 'INACTIVE':
        raise IndexError
    ecs_cluster_status = describe_ecs_cluster_response['clusters'][0]['status']
    print(f'ECS cluster \'{ecs_cluster_name}\' already exists (status: {ecs_cluster_status})')
except (IndexError, KeyError):
    create_ecs_cluster_response = ecs_client.create_cluster(clusterName=ecs_cluster_name)
    print(f'ECS cluster created. Status: {create_ecs_cluster_response["cluster"]["status"]}')

In [ ]:
# Wait for cluster to become ACTIVE
while True:
    response = ecs_client.describe_clusters(clusters=[ecs_cluster_name])
    status = response['clusters'][0]['status']
    print(f'ECS cluster status: {status}')
    if status in {'ACTIVE', 'INACTIVE', 'FAILED'}:
        break
    time.sleep(10)

### B) Create the ECS Task and deploy the container <a id='Create-the-ECS-Task-and-deploy-the-container'></a>

In [ ]:
# Register the ECS Fargate task definition
ecs_register_task_definition_response = ecs_client.register_task_definition(
    family=ecs_fargate_task_name,
    taskRoleArn=ecs_fargate_task_role,
    executionRoleArn=ecs_fargate_task_execution_role,
    networkMode='awsvpc',
    containerDefinitions=[{
        'name': ecs_container_name,
        'image': target_image_name,
        'portMappings': [{
            'containerPort': ecs_container_port,
            'hostPort': ecs_container_host_port,
            'protocol': 'tcp',
        }],
        'logConfiguration': {
            'logDriver': 'awslogs',
            'options': {
                'awslogs-create-group': 'true',
                'awslogs-region': region_name,
                'awslogs-group': f'/ecs/{ecs_fargate_task_name}',
                'awslogs-stream-prefix': 'ecs'
            }
        },
        'healthCheck': {
            'command': ecs_container_healthcheck_command_list,
            'interval': ecs_container_healthcheck_interval_in_seconds,
            'timeout': ecs_container_healthcheck_timeout_in_seconds
        }
    }],
    requiresCompatibilities=['FARGATE'],
    cpu=ecs_fargate_task_cpu,
    memory=ecs_fargate_task_memory
)

ecs_fargate_task_definition_arn = ecs_register_task_definition_response['taskDefinition']['taskDefinitionArn']
print(f'Task definition ARN: {ecs_fargate_task_definition_arn}')

In [ ]:
# Run the ECS Fargate task
ecs_run_task_response = ecs_client.run_task(
    cluster=ecs_cluster_name,
    count=ecs_fargate_task_count,
    launchType='FARGATE',
    networkConfiguration={
        'awsvpcConfiguration': {
            'subnets': ecs_fargate_task_subnet_list,
            'securityGroups': ecs_fargate_task_security_group_list,
            'assignPublicIp': 'ENABLED'
        }
    },
    taskDefinition=ecs_fargate_task_name
)

ecs_fargate_task_id = ecs_run_task_response['tasks'][0]['taskArn']
print(f'ECS Fargate Task ARN: {ecs_fargate_task_id}')

### C) Prepare to test the ECS Task <a id='Prepare-to-test-the-ECS-Task'></a>

Wait for the task to reach `RUNNING` state, then retrieve its public IP.

In [ ]:
# Wait for task to be RUNNING
while True:
    response = ecs_client.describe_tasks(cluster=ecs_cluster_name, tasks=[ecs_fargate_task_id])
    task_status = response['tasks'][0]['lastStatus']
    print(f'ECS Task status: {task_status}')
    if task_status in {'RUNNING', 'STOPPED'}:
        break
    time.sleep(5)

if task_status == 'STOPPED':
    print('ERROR: Task stopped unexpectedly.')
    stopped_reason = response['tasks'][0].get('stoppedReason', 'Unknown')
    print(f'Reason: {stopped_reason}')

In [ ]:
# Retrieve the Public IP address of the ECS Task
response = ecs_client.describe_tasks(cluster=ecs_cluster_name, tasks=[ecs_fargate_task_id])
ecs_task_attachments = response['tasks'][0]['attachments']

ecs_fargate_task_public_ip = None
for attachment in ecs_task_attachments:
    if attachment['type'] == 'ElasticNetworkInterface':
        for detail in attachment['details']:
            if detail['name'] == 'networkInterfaceId':
                eni_id = detail['value']
                eni_response = ec2_client.describe_network_interfaces(
                    NetworkInterfaceIds=[eni_id])
                ecs_fargate_task_public_ip = eni_response['NetworkInterfaces'][0]['Association']['PublicIp']

print(f'ECS Task Public IP: {ecs_fargate_task_public_ip}')
print(f'\nNOTE: Make sure your Security Group allows inbound TCP on port {ecs_container_port}')
print(f'from your IP address before testing.')

### D) Test the ECS Task <a id='Test-the-ECS-Task'></a>

Test the deployed SARIMA/ARIMA model by making HTTP POST requests.

**Request format:**
```json
{
  "response_content_type": "application/json",
  "steps": 5
}
```

**Response format:**
```json
{
  "forecast": [val1, val2, ...],
  "confidence_interval_lower": [...],
  "confidence_interval_upper": [...],
  "steps": 5
}
```

In [ ]:
# Test the health check endpoint
ecs_task_url = f'http://{ecs_fargate_task_public_ip}:{ecs_container_port}'
print(f'Testing health check...')
!curl -s {ecs_task_url}/healthcheck
print()

In [ ]:
# Test forecast — predict next 5 periods
forecast_steps = 5
request_payload = json.dumps({
    'response_content_type': 'application/json',
    'steps': forecast_steps
})

print(f'Request payload: {request_payload}')
print(f'\nResponse:')
!curl -X POST -H 'Content-Type: application/json' --data '{request_payload}' {ecs_task_url}/

In [ ]:
# Test a longer forecast (12 periods)
request_payload_12 = json.dumps({
    'response_content_type': 'application/json',
    'steps': 12
})

print(f'Request (12-step forecast): {request_payload_12}')
print(f'\nResponse:')
!curl -X POST -H 'Content-Type: application/json' --data '{request_payload_12}' {ecs_task_url}/

In [ ]:
# Test with plain text response
request_payload_text = json.dumps({
    'response_content_type': 'text/plain',
    'steps': 3
})

print(f'Request (text/plain): {request_payload_text}')
print(f'\nResponse (comma-separated forecast values):')
!curl -X POST -H 'Content-Type: application/json' --data '{request_payload_text}' {ecs_task_url}/

## 8. Cleanup <a id='Cleanup'></a>

Delete all resources created by this notebook to avoid unnecessary costs.

### A) Cleanup ECS resources <a id='Cleanup-ECS-resources'></a>

In [ ]:
# Stop the ECS Task
ecs_client.stop_task(
    cluster=ecs_cluster_name,
    task=ecs_fargate_task_id,
    reason=f'Cleanup from notebook {nb_name}'
)
print('ECS Task stopped.')

In [ ]:
# Deregister ECS Task definition
ecs_client.deregister_task_definition(taskDefinition=ecs_fargate_task_definition_arn)
print('Task definition deregistered.')

In [ ]:
# Delete the ECS cluster
ecs_client.delete_cluster(cluster=ecs_cluster_name)
print(f'ECS cluster \'{ecs_cluster_name}\' deleted.')

### B) Cleanup ECR repository <a id='Cleanup-ECR-repository'></a>

In [ ]:
# Delete the ECR private repository
try:
    ecr_client.delete_repository(repositoryName=container_image_name, force=True)
    print(f'ECR repository \'{container_image_name}\' deleted.')
except ecr_client.exceptions.RepositoryNotFoundException:
    print(f'ECR repository \'{container_image_name}\' does not exist.')

In [ ]:
# Clean up local container artifacts
import shutil
if os.path.exists(container_artifacts_dir):
    shutil.rmtree(container_artifacts_dir)
    print(f'Local artifacts directory \'{container_artifacts_dir}\' deleted.')

print('\nCleanup complete!')
print('\nNote: Docker containers/images created locally remain. Run `docker system prune` to clean them up.')